## Переобучение

### 1. Что такое переобучение

Переобучение - свойство модели терять свою обобщающую способность при обучении на ограниченной выборке данных.

Стандартный способ избежать переобучения - разделить тренеровочную выборку на обучающую и валидирующую. После прохода обучения, происходит валидация модели. 
Если качество обучения и валидации расходится - обучение лучше остановить.

Важно! Валидационная, обучающая и тестовая выборка должны отличаться. Валидация это как бы контрольная работа, обучающая выборка - материалы лекций и тестовая выборка - экзамен.

In [ ]:
import torch
import torchvision
import torch.utils.data as data
import torchvision.transforms.v2 as tfs
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision.datasets import ImageFolder


class RavelTransform(nn.Module):
    def forward(self, item):
        return item.ravel()


class DigitNN(nn.Module):
    def __init__(self, input_dim, num_hidden, output_dim):
        super().__init__()
        self.layer1 = nn.Linear(input_dim, output_dim)
        self.layer2 = nn.Linear(num_hidden, output_dim)

    def forward(self, x):
        x = self.layer1(x)
        x = nn.functional.relu(x)
        x = self.layer2(x)
        return x


model = DigitNN(28 * 28, 32, 10)

transforms = tfs.Compose([tfs.ToImage(), tfs.Grayscale,
                          tfs.ToDtype(torch.float32, scale=True),
                          RavelTransform(),])


dataset_mnist = torchvision.datasets.MNIST('datasets/mnist', download=True, train=True, transform=transforms)

# Случайным образом выбираем данные из датасета для проверки и для валидации
d_train, d_val = data.random_split(dataset_mnist, [0.7, 0.3])

# Формируем итераторы батчей, когда обучаем образы перемешиваются
train_data = data.DataLoader(d_train, batch_size=32, shuffle=True)
train_data_val = data.DataLoader(d_val, batch_size=32, shuffle=False)


optimizer = optim.Adam(params=model.parameters(), lr=0.01)
loss_function = nn.CrossEntropyLoss()
epochs = 20

model_data = torch.load('weights/mnist/model_dnn_1.tar', weights_only=True)
model.load_state_dict(model_data['model'])
transforms.load_state_dict(model_data['model'])
optimizer.load_state_dict(model_data['model'])


# Для визуализации
loss_list_val = []
loss_list = []


for _e in range(epochs):
    # Модель переводится в режим обучения в начале каждой эпохи
    model.train() 
    loss_mean = 0
    lm_count = 0

    train_tqdm = tqdm(train_data, leave=False)
    for x_train, y_train in train_tqdm:
        predict = model(x_train)
        loss = loss_function(predict, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        lm_count += 1
        loss_mean = 1/lm_count * loss.item() + (1 - 1/lm_count) * loss_mean
        train_tqdm.set_description(f'Epoch [{_e+1}/{epochs}], loss_mean={loss_mean:.3f}')

    # Валидация модели
    model.eval()
    Q_val = 0
    count_val = 0

    for x_val, y_val in train_data_val:
        with torch.no_grad():
            p = model(x_val)
            loss = loss_function(p, y_val)
            Q_val += loss.item()
            count_val += 1
    
    Q_val /= count_val

    loss_list_val.append(Q_val) # Среднее значение по валидации
    loss_list.append(loss_mean) # Среднее значение функции потерь

    print(f' | loss_mean={loss_mean:.3f}, Q_val={Q_val:.3f}')


# Создание графиков
plt.plot(loss_list)
plt.plot(loss_list_val)
plt.grid()
plt.show()

FileNotFoundError: [Errno 2] No such file or directory: 'model_dnn_1.tar'

### Когда стоит остановить обучение:

1. Расхождение показателей качества для обучающей выборки и выборки валидации;

2. От итерации к итерации (по эпохам) показатели качества Q практически не изменяются;

3. Малое изменение весовых коэффициентов;